# Go Further — Advanced Deep Learning Pipeline for Cell Classification
**Erwan Ouabdesselam**

Ce notebook présente un pipeline d'apprentissage profond avancé pour la classification de cellules sanguines en 11 classes. Il améliore le baseline ResNet50 (~75%) obtenu dans le TP2 grâce à plusieurs techniques complémentaires.

## Pipeline complet

| Technique | Rôle |
|---|---|
| **ConvNeXt-Tiny** | Backbone moderne et plus efficace que ResNet50 |
| **Focal Loss + Label Smoothing** | Meilleure gestion du déséquilibre des classes |
| **Mixup Augmentation** | Régularisation forte, frontières de décision plus douces |
| **Progressive Fine-tuning** | Fine-tuning en 2 phases pour préserver les poids pré-entraînés |
| **AdamW + Cosine Scheduler** | Convergence optimale |
| **Test Time Augmentation (TTA)** | Gain gratuit en précision à l'inférence |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pylab import rcParams
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
from torch.utils.data import DataLoader, WeightedRandomSampler

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device : {device}")

def plot_confusion(prediction, target):
    list_labels_cat = ['ARTEFACTS', 'BASOPHILES', 'BLASTES', 'EOSINOPHILES',
                       'ERYTHROBLASTES', 'LYMPHOCYTES', 'METAMYELOCYTES', 'MONOCYTES',
                       'MYELOCYTES', 'NEUTROPHILES', 'PROMYELOCYTES']
    targ_cat  = [list_labels_cat[i] for i in target]
    preds_cat = [list_labels_cat[i] for i in prediction]
    rcParams['figure.figsize'] = 8, 7
    cm    = confusion_matrix(targ_cat, preds_cat, labels=list_labels_cat, normalize='true')
    df_cm = pd.DataFrame(cm, index=list_labels_cat, columns=list_labels_cat)
    sns.heatmap(df_cm, annot=True, fmt='.2f', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Normalized Confusion Matrix')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 1. Chargement et analyse des données

On charge les labels, on encode les classes et on effectue un **split stratifié** train/test (80/20, `random_state=42`).

Le split stratifié garantit que la **distribution des 11 classes est identique** dans le train et le test, ce qui est crucial sur un dataset déséquilibré pour éviter de biaiser l'évaluation.

In [ ]:
labels = pd.read_csv('labels.csv')

le = LabelEncoder()
labels['label_encoded'] = le.fit_transform(labels['label'])
classes = le.classes_

print(f"Classes ({len(classes)}) : {list(classes)}")
print(f"Total images : {len(labels)}")

train_df, test_df = train_test_split(
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels['label_encoded']
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"\nTrain : {len(train_df)} | Test : {len(test_df)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, df, title in zip(axes, [train_df, test_df], ['Train set', 'Test set']):
    counts = df['label'].value_counts()
    ax.bar(counts.index, counts.values, color='steelblue', edgecolor='white')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Classe')
    ax.set_ylabel("Nombre d'images")
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Distribution des classes', fontsize=14)
plt.tight_layout()
plt.show()

ratio = train_df['label'].value_counts().max() / train_df['label'].value_counts().min()
print(f"Ratio max/min (déséquilibre) : {ratio:.1f}x")

## 2. Pipeline d'augmentation

L'augmentation des données est une technique de **régularisation** appliquée pendant l'entraînement : chaque image est transformée aléatoirement à chaque epoch, ce qui augmente artificiellement la diversité du dataset et réduit l'overfitting.

### Transformations et justification médicale

| Transformation | Justification |
|---|---|
| `RandomHorizontalFlip` | Les cellules n'ont pas d'orientation préférentielle |
| `RandomVerticalFlip` | Idem |
| `RandomRotation(15°)` | Légère rotation réaliste pour des cellules |
| `ColorJitter` | Variabilité de coloration entre lames (Hématoxyline/Éosine) |
| `RandomAffine` | Légères déformations de positionnement |
| `Normalize (ImageNet)` | **Obligatoire** pour utiliser les poids pré-entraînés de ConvNeXt |

On utilise un transform **train** (avec augmentations aléatoires) et un transform **test** (déterministe, sans aléatoire) pour une évaluation reproductible. Le transform **TTA** est une version intermédiaire utilisée à l'inférence.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform_train = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

transform_test = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# Transform pour TTA : légère augmentation à l'inférence
transform_tta = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

## 3. Dataset et Dataloaders

### WeightedRandomSampler

Le dataset est fortement déséquilibré (certaines classes ont jusqu'à **10× moins** d'exemples que d'autres). Sans correction, le modèle s'optimise principalement sur les classes majoritaires.

Le `WeightedRandomSampler` attribue à chaque image un poids **inversement proportionnel** à la fréquence de sa classe, de sorte que chaque batch voit les 11 classes représentées de manière équitable.

Cette approche est **complémentaire** à la Focal Loss (présentée section 5) :
- Le Sampler agit sur la **distribution des données** en entrée
- La Focal Loss agit sur la **fonction de coût** en sortie

In [ ]:
class MyDataset(torch.utils.data.Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row   = self.dataframe.iloc[index]
        path  = f"allImages_247_282/image{int(row['cytoID'])}.png"
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = int(row['label_encoded'])
        return image, label


# WeightedRandomSampler
train_labels_arr = train_df['label_encoded'].values
class_count      = np.bincount(train_labels_arr)
class_weights    = 1.0 / class_count
sample_weights   = torch.DoubleTensor(class_weights[train_labels_arr])

sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

dataset_train = MyDataset(train_df, transform=transform_train)
dataset_test  = MyDataset(test_df,  transform=transform_test)

dataloader_train = DataLoader(dataset_train, batch_size=32, sampler=sampler)
dataloader_test  = DataLoader(dataset_test,  batch_size=32, shuffle=False)

print(f"Batches train : {len(dataloader_train)} | Batches test : {len(dataloader_test)}")

## 4. Architecture : ConvNeXt-Tiny

### Pourquoi ConvNeXt plutôt que ResNet50 ?

**ConvNeXt** (Liu et al., 2022, *"A ConvNet for the 2020s"*) est une réarchitecture moderne des réseaux convolutifs, inspirée des Vision Transformers. Il conserve la structure convolutive (plus adaptée aux petits datasets que les ViT) mais intègre plusieurs innovations :

- **Dépthwise convolutions 7×7** → réceptive field plus large, capture de contexte global
- **Layer Normalization** au lieu de BatchNorm → plus stable, indépendant de la taille du batch
- **Activation GELU** au lieu de ReLU → gradient plus lisse, meilleure convergence
- **Inverted bottleneck** (comme les Transformers) → représentations plus riches

### Comparaison des backbones pour petits datasets médicaux

| Modèle | Paramètres | ImageNet Top-1 | Efficacité petits datasets |
|---|---|---|---|
| ResNet50 | 25M | 76.1% | Bonne |
| EfficientNet-B3 | 12M | 82.1% | Très bonne |
| **ConvNeXt-Tiny** | **28M** | **82.1%** | **Excellente** |

ConvNeXt-Tiny atteint les mêmes performances qu'EfficientNet-B3 avec une architecture plus homogène et des poids ImageNet mieux transférables.

### Adaptation au dataset

On remplace uniquement la dernière couche linéaire du classifieur par une couche à **11 sorties** (11 classes de cellules).

In [ ]:
def build_convnext(n_classes=11):
    model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
    # Remplacement du classifieur final (768 → 11)
    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(in_features, n_classes)
    return model

convnext_model = build_convnext().to(device)

total_params     = sum(p.numel() for p in convnext_model.parameters())
trainable_params = sum(p.numel() for p in convnext_model.parameters() if p.requires_grad)
print(f"Paramètres totaux      : {total_params:,}")
print(f"Paramètres entraînables: {trainable_params:,}")

## 5. Focal Loss + Label Smoothing

### Focal Loss

La **Cross-Entropy Loss** standard traite tous les exemples de manière égale. Sur un dataset déséquilibré, le modèle tend à s'optimiser sur les classes majoritaires et à ignorer les exemples difficiles des classes rares.

La **Focal Loss** (Lin et al., 2017 — initialement proposée pour la détection d'objets) corrige ce problème en **réduisant dynamiquement le poids des exemples faciles** :

$$FL(p_t) = -(1 - p_t)^\gamma \cdot \log(p_t)$$

- Quand $p_t \approx 1$ (exemple facile) : $(1 - p_t)^\gamma \approx 0$ → contribution quasi nulle
- Quand $p_t \approx 0$ (exemple difficile) : $(1 - p_t)^\gamma \approx 1$ → perte normale

On utilise **γ = 2** (valeur standard de la littérature).

### Label Smoothing

Le **label smoothing** remplace les labels durs {0, 1} par des labels mous {ε/(K-1), 1-ε} avec ε = 0.1. Cela réduit l'overconfidence du modèle et améliore la généralisation, en particulier sur les classes morphologiquement proches.

In [ ]:
class FocalLoss(nn.Module):

    def __init__(self, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.gamma           = gamma
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        # Cross-entropy avec label smoothing
        ce_loss = F.cross_entropy(inputs, targets, reduction='none',
                                  label_smoothing=self.label_smoothing)
        # Probabilité de la classe prédite correctement
        pt = torch.exp(-ce_loss)
        # Focal weight : pénalise moins les exemples faciles
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()


criterion = FocalLoss(gamma=2.0, label_smoothing=0.1)

# Test de cohérence
dummy_inputs  = torch.randn(4, 11)
dummy_targets = torch.randint(0, 11, (4,))
print(f"Focal Loss test : {criterion(dummy_inputs, dummy_targets).item():.4f} ✓")

## 6. Mixup Augmentation

Le **Mixup** (Zhang et al., 2018) est une technique de régularisation qui crée des exemples d'entraînement synthétiques en **interpolant linéairement** des paires d'images et leurs labels :

$$\tilde{x} = \lambda \cdot x_i + (1 - \lambda) \cdot x_j, \quad \tilde{y} = \lambda \cdot y_i + (1 - \lambda) \cdot y_j$$

où $\lambda \sim \text{Beta}(\alpha, \alpha)$, avec **α = 0.4**.

### Avantages

- Force le modèle à apprendre des **frontières de décision linéaires** entre classes
- Réduit l'overconfidence (complémentaire au label smoothing)
- Particulièrement utile pour les classes biologiquement proches : **BLASTES ↔ MYELOCYTES**, **METAMYELOCYTES ↔ NEUTROPHILES**
- Régularisation implicite sans paramètres supplémentaires

### Implémentation

Le Mixup est appliqué **au niveau du batch** pendant la phase 2 (fine-tuning complet). La loss est calculée comme une combinaison convexe des deux pertes :

$$\mathcal{L} = \lambda \cdot \mathcal{L}(f(\tilde{x}), y_i) + (1 - \lambda) \cdot \mathcal{L}(f(\tilde{x}), y_j)$$

In [ ]:
def mixup_data(x, y, alpha=0.4):
    """Applique Mixup à un batch — retourne le batch mixé et les deux labels."""
    lam   = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    index = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    return mixed_x, y, y[index], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Calcule la perte Mixup comme combinaison convexe."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

## 7. Progressive Fine-tuning

Lorsqu'on fine-tune un modèle pré-entraîné, entraîner **toutes les couches simultanément** dès le début avec un LR élevé peut détruire les représentations ImageNet apprises par les couches profondes du backbone.

### Stratégie en 2 phases

**Phase 1 — Backbone gelé (5 epochs)**
- Seul le **classifieur final** est entraîné (faible nombre de paramètres)
- Le backbone conserve ses poids ImageNet
- LR plus élevé (1e-3) car gradient ne traverse pas le backbone
- **Objectif** : initialiser correctement le classifieur avant tout fine-tuning profond

**Phase 2 — Fine-tuning complet + Mixup (15 epochs)**
- **Toutes les couches** sont dégelées
- LR beaucoup plus faible (1e-4) pour ne pas perturber les représentations
- **Cosine Annealing Scheduler** : le LR décroît doucement de 1e-4 à 1e-6
- **Mixup activé** pour régulariser le fine-tuning
- **Objectif** : affiner toutes les représentations pour les données médicales

Cette approche est plus stable qu'un fine-tuning direct et donne systématiquement de **meilleurs résultats**.

In [ ]:
def freeze_backbone(model):
    """Gèle toutes les couches sauf le classifieur."""
    for name, param in model.named_parameters():
        if 'classifier' not in name:
            param.requires_grad = False
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Backbone gelé     — Paramètres entraînables : {n:,}")


def unfreeze_backbone(model):
    """Dégèle toutes les couches."""
    for param in model.parameters():
        param.requires_grad = True
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Backbone dégelé   — Paramètres entraînables : {n:,}")


def train_epoch(model, dataloader, criterion, optimizer, use_mixup=False):
    model.train()
    running_loss = 0.0
    for images, labels_batch in tqdm(dataloader, leave=False):
        images, labels_batch = images.to(device), labels_batch.to(device)
        if use_mixup:
            images, y_a, y_b, lam = mixup_data(images, labels_batch)
            optimizer.zero_grad()
            loss = mixup_criterion(criterion, model(images), y_a, y_b, lam)
        else:
            optimizer.zero_grad()
            loss = criterion(model(images), labels_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    return running_loss / len(dataloader)


def eval_epoch(model, dataloader, criterion):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for images, labels_batch in dataloader:
            images, labels_batch = images.to(device), labels_batch.to(device)
            running_loss += criterion(model(images), labels_batch).item()
    return running_loss / len(dataloader)

### Phase 1 — Entraînement du classifieur seul

In [ ]:
print("=" * 55)
print("  PHASE 1 — Classifieur seul (backbone gelé)")
print("=" * 55)

EPOCHS_P1 = 5
LR_P1     = 1e-3

freeze_backbone(convnext_model)

optimizer_p1 = optim.AdamW(
    filter(lambda p: p.requires_grad, convnext_model.parameters()),
    lr=LR_P1, weight_decay=0.01
)
scheduler_p1 = optim.lr_scheduler.CosineAnnealingLR(optimizer_p1, T_max=EPOCHS_P1, eta_min=1e-5)

train_losses_p1, val_losses_p1 = [], []

for epoch in range(EPOCHS_P1):
    tr  = train_epoch(convnext_model, dataloader_train, criterion, optimizer_p1, use_mixup=False)
    val = eval_epoch(convnext_model, dataloader_test, criterion)
    scheduler_p1.step()
    train_losses_p1.append(tr)
    val_losses_p1.append(val)
    print(f"  Epoch {epoch+1}/{EPOCHS_P1} — Train: {tr:.4f} | Val: {val:.4f} | LR: {scheduler_p1.get_last_lr()[0]:.6f}")

### Phase 2 — Fine-tuning complet avec Mixup

In [ ]:
print("=" * 55)
print("  PHASE 2 — Fine-tuning complet + Mixup")
print("=" * 55)

EPOCHS_P2 = 15
LR_P2     = 1e-4

unfreeze_backbone(convnext_model)

optimizer_p2 = optim.AdamW(convnext_model.parameters(), lr=LR_P2, weight_decay=0.01)
scheduler_p2 = optim.lr_scheduler.CosineAnnealingLR(optimizer_p2, T_max=EPOCHS_P2, eta_min=1e-6)

train_losses_p2, val_losses_p2 = [], []

for epoch in range(EPOCHS_P2):
    tr  = train_epoch(convnext_model, dataloader_train, criterion, optimizer_p2, use_mixup=True)
    val = eval_epoch(convnext_model, dataloader_test, criterion)
    scheduler_p2.step()
    train_losses_p2.append(tr)
    val_losses_p2.append(val)
    print(f"  Epoch {epoch+1}/{EPOCHS_P2} — Train: {tr:.4f} | Val: {val:.4f} | LR: {scheduler_p2.get_last_lr()[0]:.6f}")

In [ ]:
# Courbes de perte
all_train = train_losses_p1 + train_losses_p2
all_val   = val_losses_p1   + val_losses_p2
epochs_x  = list(range(1, EPOCHS_P1 + EPOCHS_P2 + 1))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(epochs_x, all_train, label='Train Loss', marker='o', linewidth=2)
ax.plot(epochs_x, all_val,   label='Val Loss',   marker='s', linewidth=2)
ax.axvline(x=EPOCHS_P1 + 0.5, color='red', linestyle='--', alpha=0.7,
           label=f'Phase 1 → Phase 2 (Mixup activé)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Focal Loss')
ax.set_title('Courbes de perte — Fine-tuning progressif ConvNeXt-Tiny')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Évaluation avec Test Time Augmentation (TTA)

Le **Test Time Augmentation** consiste à appliquer plusieurs augmentations aléatoires à chaque image de test, puis à **moyenner les probabilités** (softmax) sur toutes les passes.

**Pourquoi ça marche ?**
Chaque augmentation produit une vue légèrement différente de l'image. La moyenne des probabilités réduit la **variance des prédictions** et donne une estimation plus robuste de la distribution de classe.

**Coût** : N × temps d'inférence standard (ici N = 8), mais **aucun ré-entraînement**.

On compare les résultats avec et sans TTA pour quantifier le gain.

In [ ]:
def find_accuracy(model, dataloader):
    """Évaluation standard (sans TTA)."""
    model.eval()
    correct, total = 0, 0
    label_list, pred_list = [], []
    with torch.no_grad():
        for images, labels_batch in dataloader:
            images, labels_batch = images.to(device), labels_batch.to(device)
            _, predicted = torch.max(model(images).data, 1)
            label_list += labels_batch.tolist()
            pred_list  += predicted.tolist()
            total   += labels_batch.size(0)
            correct += (predicted == labels_batch).sum().item()
    return pred_list, label_list, correct / total


def find_accuracy_tta(model, dataframe, n_tta=8):
    """Évaluation avec Test Time Augmentation."""
    model.eval()
    all_probs = []
    for _ in range(n_tta):
        loader = DataLoader(MyDataset(dataframe, transform=transform_tta), batch_size=32, shuffle=False)
        probs_list = []
        with torch.no_grad():
            for images, _ in loader:
                probs = torch.softmax(model(images.to(device)), dim=1)
                probs_list.append(probs.cpu())
        all_probs.append(torch.cat(probs_list))
    mean_probs  = torch.stack(all_probs).mean(0)
    predictions = mean_probs.argmax(dim=1).tolist()
    true_labels = dataframe['label_encoded'].tolist()
    accuracy    = sum(p == t for p, t in zip(predictions, true_labels)) / len(true_labels)
    return predictions, true_labels, accuracy


# Évaluation standard
pred_std, target_std, acc_std = find_accuracy(convnext_model, dataloader_test)
print(f"Accuracy standard     : {acc_std*100:.2f}%")

# Évaluation avec TTA
print("Calcul TTA (8 passes)...")
pred_tta, target_tta, acc_tta = find_accuracy_tta(convnext_model, test_df, n_tta=8)
print(f"Accuracy avec TTA     : {acc_tta*100:.2f}%")
print(f"Gain TTA              : +{(acc_tta - acc_std)*100:.2f}%")

### Résultats — Matrices de confusion

In [ ]:
print("=== Matrice de confusion — Standard ===")
plot_confusion(pred_std, target_std)

print("=== Matrice de confusion — Avec TTA (8 passes) ===")
plot_confusion(pred_tta, target_tta)

In [ ]:
print("=== Rapport de classification — ConvNeXt-Tiny + TTA ===\n")
print(classification_report(target_tta, pred_tta, target_names=classes))

## 9. Comparaison des modèles

In [ ]:
print("=" * 62)
print("            COMPARAISON COMPLÈTE DES MODÈLES")
print("=" * 62)
print(f"  CNN from scratch (TP2 Q8)                  : ~60%")
print(f"  ResNet50 pré-entraîné (TP2 Q11)            : ~75%")
print(f"  ResNet50 + WeightedSampler (TP2 Q12)       : ~73%")
print(f"  {'─'*48}")
print(f"  ConvNeXt-Tiny + Focal Loss  (standard)     : {acc_std*100:.1f}%")
print(f"  ConvNeXt-Tiny + Focal Loss  (+ TTA x8)     : {acc_tta*100:.1f}%")
print("=" * 62)

## 10. Discussion & Analyse

### Apport de chaque composante du pipeline

**ConvNeXt-Tiny vs ResNet50**
ConvNeXt utilise des convolutions dépthwise 7×7 et une Layer Normalization, ce qui lui permet de capturer un contexte spatial plus large à chaque couche. Ses poids ImageNet transfèrent mieux sur les images médicales grâce à une réceptive field plus grande et une architecture proche des Transformers.

**Focal Loss + Label Smoothing**
Sur ce dataset déséquilibré, la Focal Loss concentre l'apprentissage sur les classes difficiles (BLASTES, METAMYELOCYTES, BASOPHILES) plutôt que d'optimiser les classes majoritaires. Le label smoothing réduit l'overconfidence et améliore la robustesse du classifieur.

**Progressive Fine-tuning**
La Phase 1 (backbone gelé) initialise correctement le classifieur avant de propager des gradients dans le backbone. La Phase 2 (avec Mixup) affine ensuite toutes les représentations de manière régularisée, sans détruire les features ImageNet.

**Mixup**
Particulièrement bénéfique pour les classes morphologiquement proches (BLASTES/MYELOCYTES, METAMYELOCYTES/NEUTROPHILES) dont les frontières biologiques sont floues. Le Mixup force le modèle à apprendre des représentations interpolables plutôt que des frontières rigides.

**AdamW + Cosine Annealing**
AdamW avec weight decay explicite évite la dégradation des poids lors du fine-tuning. Le Cosine Scheduler permet une descente de gradient progressive qui affine les poids sans osciller.

**Test Time Augmentation**
Gain obtenu sans ré-entraînement, par simple agrégation de prédictions stochastiques. Particulièrement efficace sur les cellules où l'orientation n'est pas informative.

### Limites et perspectives

- Un **ensemble de modèles** (ConvNeXt + EfficientNet + ResNet50) donnerait encore de meilleures performances en combinant des représentations complémentaires
- L'intégration des **features morphologiques du TP1** (surface, périmètre, intensités Hématoxyline/Éosine) via une architecture multimodale pourrait apporter un signal complémentaire non capturé par le CNN
- Une **normalisation de la coloration** (Stain Normalization, ex. Macenko) spécifique à l'histologie permettrait de réduire la variabilité inter-lames qui pollue actuellement l'apprentissage